In [ ]:
import nltk
import os
from docling.document_converter import DocumentConverter
from nltk.tokenize import sent_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification, Pipeline

cache_path = os.path.abspath("./.cache/")


def get_text(path: str) -> str:
    converter = DocumentConverter()
    result = converter.convert(path)
    return result.document.export_to_text()


def get_sentences(text: str) -> list[str]:
    nltk.download("punkt_tab", download_dir=cache_path)
    if cache_path not in nltk.data.path:
        nltk.data.path.append(cache_path)
    return sent_tokenize(text)


def get_pipeline() -> Pipeline:
    tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER", cache_dir=cache_path)
    model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER", cache_dir=cache_path)
    return pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple", stride=128)


def annotate(sentence: str, entities: list[dict]) -> str:
    annotated = sentence
    for entity in entities:
        label = entity["entity_group"]
        start = entity["start"]
        end = entity["end"]
        annotated = annotated[:start] + f"[{label}]" + annotated[start:end] + f"[/{label}]" + annotated[end:]
    return annotated

In [ ]:
def main():
    text = get_text("data/obama.pdf")
    text = text[text.find("\n\n") + 2:]  # remove title
    sentences = get_sentences(text)
    pipe = get_pipeline()
    for sentence in sentences:
        entities = sorted(pipe(sentence), key=lambda x: x["start"], reverse=True)
        print(f"Input: {sentence}")
        print(f"Output: {entities}")
        print(f"Annotated: {annotate(sentence, entities)}")
        print()


main()